In [1]:
import argparse
import json
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
import snntorch as snn


### Config

In [2]:
SEED = 42

WINDOW = 96          # 24 hours at 15-minute resolution
HORIZON = 1          # predict next 15-minute interval
STEP_SECONDS = 15 * 60

TRAIN_FRAC = 0.70
VAL_FRAC = 0.15

BATCH_SIZE = 256
EPOCHS = 40
PATIENCE = 6         # early stopping, in epochs with no val-RMSE improvement
LR = 1e-3
WEIGHT_DECAY = 1e-4

HIDDEN1 = 64
HIDDEN2 = 32
BETA_INIT = 0.90
DROPOUT = 0.15

HOME_EMBED_DIM = 4

# Weighted Huber: weight = 1 + PEAK_WEIGHT_ALPHA * |scaled target delta|
PEAK_WEIGHT_ALPHA = 2.0

NUM_WORKERS = 0

FEATURES = [
    "grid",
    "air1",
    "furnace1",
    "solar",
    "refrigerator1",
    "car1",
    "waterheater1",
    "drye1",
    "dishwasher1",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
]

TARGET = "grid"

MIN_COVERAGE = 0.90

RESULTS_DIR = Path("results/snn_v3")
PLOTS_DIR = RESULTS_DIR / "plots"


# Appliance circuits: measured power draw, should never be negative.
APPLIANCE_COLS = ["air1", "furnace1", "refrigerator1", "car1", "waterheater1", "drye1", "dishwasher1"]

# Signed columns: grid can be negative (import/export), solar generation
# is also stored signed -- neither gets clipped for negativity.
SIGNED_COLS = ["grid", "solar"]

RAW_NUMERIC_COLS = SIGNED_COLS + APPLIANCE_COLS
REQUIRED_COLS = ["grid", "air1", "furnace1"]
OPTIONAL_COLS = [c for c in RAW_NUMERIC_COLS if c not in REQUIRED_COLS]

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


### Cleaning

In [3]:

def clean_home(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["local_15min"] = pd.to_datetime(df["local_15min"], errors="coerce", utc=True)

    df = df.dropna(subset=["local_15min"])
    df = df.sort_values("local_15min")
    df = df.drop_duplicates("local_15min")

    for col in RAW_NUMERIC_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Appliance measurements should not be negative. Grid and solar are
    # allowed to be (import/export, generation sign convention).
    for col in APPLIANCE_COLS:
        df.loc[df[col] < 0, col] = np.nan

    local_time = df["local_15min"].dt.tz_convert("America/Chicago")

    hour = local_time.dt.hour + local_time.dt.minute / 60.0
    dow = local_time.dt.dayofweek

    df["hour_sin"] = np.sin(2 * np.pi * hour / 24.0)
    df["hour_cos"] = np.cos(2 * np.pi * hour / 24.0)
    df["dow_sin"] = np.sin(2 * np.pi * dow / 7.0)
    df["dow_cos"] = np.cos(2 * np.pi * dow / 7.0)

    return df


def coverage(df: pd.DataFrame, col: str) -> float:
    return df[col].notna().mean()



In [4]:
def prepare_homes(raw: pd.DataFrame):

    homes = []

    for dataid, home in raw.groupby("dataid"):

        home = clean_home(home)

        if len(home) < WINDOW + HORIZON + 100:
            continue
        if any(coverage(home, col) < MIN_COVERAGE for col in REQUIRED_COLS):
            continue

        homes.append((dataid, home))

    print(f"\nHomes retained: {len(homes)}")

    for dataid, home in homes:
        cov_str = " | ".join(
            f"{col}={coverage(home, col):.1%}" for col in RAW_NUMERIC_COLS
        )
        print(f"  {dataid}: {len(home):,} rows | {cov_str}")

    return homes



In [5]:
def get_train_bounds(train_parts):

    combined = pd.concat(train_parts, ignore_index=True)

    bounds = {}
    for col in APPLIANCE_COLS:
        s = combined[col].dropna()
        if len(s) == 0:
            continue
        bounds[col] = (float(s.quantile(0.001)), float(s.quantile(0.999)))

    return bounds


def apply_cleaning_and_imputation(df, bounds):

    df = df.copy()

    for col, (lo, hi) in bounds.items():
        df.loc[(df[col] < lo) | (df[col] > hi), col] = np.nan

    for col in RAW_NUMERIC_COLS:
        df[col] = df[col].interpolate(method="linear", limit=4, limit_direction="both")

    # Remaining appliance gaps -> zero (homes already cleared >=90% coverage).
    for col in APPLIANCE_COLS:
        df[col] = df[col].fillna(0.0)

    # Solar: leftover gaps (e.g. nighttime) legitimately read ~0.
    df["solar"] = df["solar"].fillna(0.0)

    df = df.dropna(subset=["grid"])

    return df



In [6]:
def build_home_index(homes):
    return {dataid: idx for idx, (dataid, _) in enumerate(homes)}


### Dataset

In [7]:
class WindowDataset(Dataset):

    def __init__(self, homes, feature_scaler, delta_scaler, home_to_idx):

        self.X = []
        self.y = []            # scaled delta target
        self.last_raw = []     # unscaled grid value at the last input step
        self.home_idx = []

        expected_span = pd.Timedelta(seconds=STEP_SECONDS * WINDOW)

        for dataid, df in homes:

            x = df[FEATURES].to_numpy(dtype=np.float32)
            grid_raw = df[TARGET].to_numpy(dtype=np.float32)
            times = df["local_15min"].to_numpy()

            x_scaled = feature_scaler.transform(x)

            n = len(df)
            hidx = home_to_idx[dataid]

            for end in range(WINDOW, n - HORIZON + 1):

                start = end - WINDOW
                target_idx = end + HORIZON - 1

                if target_idx >= n:
                    break

                # Gap check: the window + the target step should span
                # exactly WINDOW * 15 minutes of real elapsed time.
                span = pd.Timestamp(times[target_idx]) - pd.Timestamp(times[start])
                if span != expected_span:
                    continue

                delta = grid_raw[target_idx] - grid_raw[end - 1]

                self.X.append(x_scaled[start:end])
                self.y.append(delta)
                self.last_raw.append(grid_raw[end - 1])
                self.home_idx.append(hidx)

        self.X = np.asarray(self.X, dtype=np.float32)
        self.y = delta_scaler.transform(
            np.asarray(self.y, dtype=np.float32).reshape(-1, 1)
        ).reshape(-1).astype(np.float32)
        self.last_raw = np.asarray(self.last_raw, dtype=np.float32)
        self.home_idx = np.asarray(self.home_idx, dtype=np.int64)

        print(f"Dataset: {len(self.X):,} sequences (gap-skipped windows excluded)")

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.X[idx]),
            torch.tensor(self.y[idx], dtype=torch.float32),
            torch.tensor(self.last_raw[idx], dtype=torch.float32),
            torch.tensor(self.home_idx[idx], dtype=torch.long),
        )


In [8]:
def collect_train_deltas(homes):
    """Raw (unscaled) next-step deltas from the train split, used only to
    fit the delta target scaler."""
    deltas = []
    for _, df in homes:
        grid_raw = df[TARGET].to_numpy(dtype=np.float32)
        deltas.append(grid_raw[1:] - grid_raw[:-1])
    return np.concatenate(deltas).reshape(-1, 1)



### Model

In [9]:
class SNNRegressor(nn.Module):

    def __init__(
        self,
        input_size,
        num_homes,
        hidden1=HIDDEN1,
        hidden2=HIDDEN2,
        beta_init=BETA_INIT,
        home_embed_dim=HOME_EMBED_DIM,
        dropout=DROPOUT,
    ):
        super().__init__()

        self.home_embedding = nn.Embedding(num_homes, home_embed_dim)

        fused_input_size = input_size + home_embed_dim

        self.fc1 = nn.Linear(fused_input_size, hidden1)
        self.lif1 = snn.Leaky(
            beta=beta_init, learn_beta=True, spike_grad=snn.surrogate.fast_sigmoid()
        )
        self.drop1 = nn.Dropout(dropout)

        self.fc2 = nn.Linear(hidden1, hidden2)
        self.lif2 = snn.Leaky(
            beta=beta_init, learn_beta=True, spike_grad=snn.surrogate.fast_sigmoid()
        )
        self.drop2 = nn.Dropout(dropout)

        self.fc3 = nn.Linear(hidden2, 1)

        # Non-spiking leaky readout for regression, learnable decay.
        self.readout = snn.Leaky(
            beta=beta_init, learn_beta=True, reset_mechanism="none", output=True
        )

    def forward(self, x, home_idx):

        batch_size = x.size(0)

        home_vec = self.home_embedding(home_idx)                 # (B, E)
        home_vec = home_vec.unsqueeze(1).expand(-1, x.size(1), -1)  # (B, T, E)
        x = torch.cat([x, home_vec], dim=-1)

        mem1 = torch.zeros(batch_size, self.fc1.out_features, device=x.device)
        mem2 = torch.zeros(batch_size, self.fc2.out_features, device=x.device)
        mem3 = torch.zeros(batch_size, 1, device=x.device)

        for t in range(x.size(1)):

            cur1 = self.fc1(x[:, t, :])
            spk1, mem1 = self.lif1(cur1, mem1)
            spk1 = self.drop1(spk1)

            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            spk2 = self.drop2(spk2)

            cur3 = self.fc3(spk2)
            _, mem3 = self.readout(cur3, mem3)

        # Predicted (scaled) delta for the next step.
        return mem3.squeeze(-1)



In [10]:
def weighted_huber(pred, target, delta=1.0, alpha=PEAK_WEIGHT_ALPHA):
    """Huber loss with per-sample weight that grows with target magnitude,
    so large transitions aren't drowned out by many small near-zero ones."""
    err = pred - target
    abs_err = err.abs()
    quadratic = torch.clamp(abs_err, max=delta)
    linear = abs_err - quadratic
    base = 0.5 * quadratic ** 2 + delta * linear

    weight = 1.0 + alpha * target.abs()
    return (base * weight).mean()



### Train Loop

In [11]:
def train_one_epoch(model, loader, optimizer, device):

    model.train()
    total_loss = 0.0

    for X, y, _, home_idx in loader:

        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        home_idx = home_idx.to(device, non_blocking=True)

        optimizer.zero_grad()

        pred = model(X, home_idx)
        loss = weighted_huber(pred, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * len(X)

    return total_loss / len(loader.dataset)


### Evaluation

In [12]:
@torch.no_grad()
def predict(model, loader, delta_scaler, device):

    model.eval()

    pred_deltas = []
    actual_deltas = []
    last_raws = []

    for X, y, last_raw, home_idx in loader:

        X = X.to(device, non_blocking=True)
        home_idx = home_idx.to(device, non_blocking=True)

        pred = model(X, home_idx)

        pred_deltas.append(pred.cpu().numpy())
        actual_deltas.append(y.numpy())
        last_raws.append(last_raw.numpy())

    pred_deltas = np.concatenate(pred_deltas)
    actual_deltas = np.concatenate(actual_deltas)
    last_raws = np.concatenate(last_raws)

    pred_deltas_raw = delta_scaler.inverse_transform(
        pred_deltas.reshape(-1, 1)
    ).reshape(-1)
    actual_deltas_raw = delta_scaler.inverse_transform(
        actual_deltas.reshape(-1, 1)
    ).reshape(-1)

    preds_abs = last_raws + pred_deltas_raw
    actuals_abs = last_raws + actual_deltas_raw

    return actuals_abs, preds_abs


In [13]:
def calculate_metrics(actuals, preds):
    return {
        "MAE": float(mean_absolute_error(actuals, preds)),
        "RMSE": float(math.sqrt(mean_squared_error(actuals, preds))),
        "R2": float(r2_score(actuals, preds)),
    }



In [14]:
def save_training_plots(history):

    plt.figure(figsize=(10, 5))
    plt.plot(history["epoch"], history["train_loss"], label="Train loss")
    plt.xlabel("Epoch")
    plt.ylabel("Weighted Huber loss (delta scale)")
    plt.title("SNN v2 Training Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "training_loss.png", dpi=160)
    plt.close()

    plt.figure(figsize=(10, 5))
    plt.plot(history["epoch"], history["val_rmse"], label="Validation RMSE")
    plt.xlabel("Epoch")
    plt.ylabel("RMSE (W)")
    plt.title("SNN v2 Validation RMSE (absolute grid)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "validation_rmse.png", dpi=160)
    plt.close()

    plt.figure(figsize=(10, 5))
    plt.plot(history["epoch"], history["val_mae"], label="Validation MAE")
    plt.xlabel("Epoch")
    plt.ylabel("MAE (W)")
    plt.title("SNN v2 Validation MAE (absolute grid)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "validation_mae.png", dpi=160)
    plt.close()


def save_test_plots(actuals, preds):

    n = min(1000, len(actuals))

    plt.figure(figsize=(12, 6))
    plt.plot(actuals[:n], label="Actual")
    plt.plot(preds[:n], label="SNN v2 prediction")
    plt.xlabel("Test sample")
    plt.ylabel("Grid load")
    plt.title(f"SNN v2: Actual vs Predicted (first {n} test samples)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "test_actual_vs_pred.png", dpi=160)
    plt.close()

    residuals = preds - actuals

    plt.figure(figsize=(10, 5))
    plt.hist(residuals, bins=100)
    plt.axvline(0, linestyle="--")
    plt.xlabel("Prediction error (W)")
    plt.ylabel("Count")
    plt.title("SNN v2 Test Residuals")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "test_residuals.png", dpi=160)
    plt.close()


In [15]:
# Load and prepare the data.
CSV_PATH = next(
    path for path in (
        Path("../data/15minute_data_austin/15minute_data_austin.csv"),
        Path("data/15minute_data_austin/15minute_data_austin.csv"),
    )
    if path.exists()
)

seed_everything(SEED)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
print(f"Loading: {CSV_PATH}")
raw = pd.read_csv(CSV_PATH, low_memory=False)
print(f"Loaded {len(raw):,} rows")

homes = prepare_homes(raw)
if not homes:
    raise RuntimeError("No homes satisfy the coverage requirements.")

home_to_idx = build_home_index(homes)
num_homes = len(home_to_idx)
train_homes, val_homes, test_homes = [], [], []

for dataid, home in homes:
    n = len(home)
    train_end = int(n * TRAIN_FRAC)
    val_end = int(n * (TRAIN_FRAC + VAL_FRAC))
    train_homes.append((dataid, home.iloc[:train_end].copy()))
    val_homes.append((dataid, home.iloc[train_end:val_end].copy()))
    test_homes.append((dataid, home.iloc[val_end:].copy()))

bounds = get_train_bounds([df for _, df in train_homes])
train_homes = [(dataid, apply_cleaning_and_imputation(df, bounds)) for dataid, df in train_homes]
val_homes = [(dataid, apply_cleaning_and_imputation(df, bounds)) for dataid, df in val_homes]
test_homes = [(dataid, apply_cleaning_and_imputation(df, bounds)) for dataid, df in test_homes]

Device: cuda
Loading: ../data/15minute_data_austin/15minute_data_austin.csv
Loaded 873,286 rows

Homes retained: 15
  661: 35,032 rows | grid=99.1% | solar=99.1% | air1=98.2% | furnace1=92.5% | refrigerator1=92.5% | car1=97.1% | waterheater1=0.0% | drye1=73.6% | dishwasher1=92.0%
  1642: 34,648 rows | grid=99.5% | solar=99.5% | air1=99.5% | furnace1=96.6% | refrigerator1=96.6% | car1=75.6% | waterheater1=0.0% | drye1=28.8% | dishwasher1=96.6%
  2335: 34,468 rows | grid=100.0% | solar=100.0% | air1=97.0% | furnace1=100.0% | refrigerator1=100.0% | car1=100.0% | waterheater1=0.0% | drye1=100.0% | dishwasher1=97.4%
  2818: 34,980 rows | grid=100.0% | solar=100.0% | air1=92.4% | furnace1=99.9% | refrigerator1=100.0% | car1=0.0% | waterheater1=0.0% | drye1=0.0% | dishwasher1=100.0%
  3456: 34,932 rows | grid=100.0% | solar=100.0% | air1=100.0% | furnace1=100.0% | refrigerator1=100.0% | car1=0.0% | waterheater1=0.0% | drye1=55.1% | dishwasher1=99.9%
  6139: 35,036 rows | grid=100.0% | solar=1

In [16]:
# Fit train-only scalers and build datasets/loaders.
feature_scaler = StandardScaler()
delta_scaler = StandardScaler()
feature_scaler.fit(pd.concat([df[FEATURES] for _, df in train_homes], ignore_index=True))
delta_scaler.fit(collect_train_deltas(train_homes))

train_dataset = WindowDataset(train_homes, feature_scaler, delta_scaler, home_to_idx)
val_dataset = WindowDataset(val_homes, feature_scaler, delta_scaler, home_to_idx)
test_dataset = WindowDataset(test_homes, feature_scaler, delta_scaler, home_to_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

/home/ahaandesai/Dev/FYP/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/ahaandesai/Dev/FYP/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/ahaandesai/Dev/FYP/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/ahaandesai/Dev/FYP/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/ahaandesai/Dev/FYP/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScal

Dataset: 359,862 sequences (gap-skipped windows excluded)


/home/ahaandesai/Dev/FYP/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/ahaandesai/Dev/FYP/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/ahaandesai/Dev/FYP/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/ahaandesai/Dev/FYP/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/ahaandesai/Dev/FYP/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScal

Dataset: 75,187 sequences (gap-skipped windows excluded)


/home/ahaandesai/Dev/FYP/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/ahaandesai/Dev/FYP/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/ahaandesai/Dev/FYP/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/ahaandesai/Dev/FYP/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/ahaandesai/Dev/FYP/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScal

Dataset: 77,070 sequences (gap-skipped windows excluded)


In [17]:
# Initialize the model and optimizer.
model = SNNRegressor(input_size=len(FEATURES), num_homes=num_homes).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3,
)

history = {
    "epoch": [],
    "train_loss": [],
    "val_mae": [],
    "val_rmse": [],
    "val_r2": [],
    "lr": [],
}
best_val = float("inf")
best_state = None
epochs_since_improvement = 0

In [18]:
# Train with validation-based early stopping.
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_actuals, val_preds = predict(model, val_loader, delta_scaler, device)
    val_metrics = calculate_metrics(val_actuals, val_preds)
    scheduler.step(val_metrics["RMSE"])
    lr = optimizer.param_groups[0]["lr"]

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_mae"].append(val_metrics["MAE"])
    history["val_rmse"].append(val_metrics["RMSE"])
    history["val_r2"].append(val_metrics["R2"])
    history["lr"].append(lr)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | Loss {train_loss:.5f} | "
        f"Val MAE {val_metrics['MAE']:.4f} | "
        f"Val RMSE {val_metrics['RMSE']:.4f} | "
        f"Val R2 {val_metrics['R2']:.4f}"
    )

    if val_metrics["RMSE"] < best_val - 1e-5:
        best_val = val_metrics["RMSE"]
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }
        epochs_since_improvement = 0
    else:
        epochs_since_improvement += 1
        if epochs_since_improvement >= PATIENCE:
            print(f"Early stopping at epoch {epoch}.")
            break

if best_state is not None:
    model.load_state_dict(best_state)

Epoch 01/40 | Loss 1.77076 | Val MAE 0.6313 | Val RMSE 1.0128 | Val R2 0.5872
Epoch 02/40 | Loss 1.65176 | Val MAE 0.6601 | Val RMSE 0.9885 | Val R2 0.6068
Epoch 03/40 | Loss 1.52030 | Val MAE 0.7086 | Val RMSE 0.9930 | Val R2 0.6032
Epoch 04/40 | Loss 1.46944 | Val MAE 0.7173 | Val RMSE 0.9901 | Val R2 0.6055
Epoch 05/40 | Loss 1.42802 | Val MAE 0.7031 | Val RMSE 0.9870 | Val R2 0.6080
Epoch 06/40 | Loss 1.39400 | Val MAE 0.6995 | Val RMSE 0.9732 | Val R2 0.6189
Epoch 07/40 | Loss 1.36105 | Val MAE 0.6845 | Val RMSE 0.9707 | Val R2 0.6208
Epoch 08/40 | Loss 1.34213 | Val MAE 0.6746 | Val RMSE 0.9731 | Val R2 0.6189
Epoch 09/40 | Loss 1.31843 | Val MAE 0.6664 | Val RMSE 0.9613 | Val R2 0.6281
Epoch 10/40 | Loss 1.29429 | Val MAE 0.6713 | Val RMSE 0.9747 | Val R2 0.6177
Epoch 11/40 | Loss 1.29513 | Val MAE 0.6501 | Val RMSE 0.9617 | Val R2 0.6278
Epoch 12/40 | Loss 1.28028 | Val MAE 0.6704 | Val RMSE 0.9725 | Val R2 0.6194
Epoch 13/40 | Loss 1.26750 | Val MAE 0.6562 | Val RMSE 0.9631 | 

In [19]:
# Evaluate the best model on the held-out test set.
test_actuals, test_preds = predict(model, test_loader, delta_scaler, device)
test_metrics = calculate_metrics(test_actuals, test_preds)

print(f"Test MAE: {test_metrics['MAE']:.4f}")
print(f"Test RMSE: {test_metrics['RMSE']:.4f}")
print(f"Test R2: {test_metrics['R2']:.4f}")

Test MAE: 0.4295
Test RMSE: 0.6904
Test R2: 0.7464


In [20]:
# Save the model, metrics, history, predictions, and plots.
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "feature_scaler": feature_scaler,
        "delta_scaler": delta_scaler,
        "home_to_idx": home_to_idx,
        "features": FEATURES,
        "target": TARGET,
        "window": WINDOW,
        "horizon": HORIZON,
    },
    RESULTS_DIR / "model.pt",
)

metrics = {
    "model": "SNN_v2_delta",
    "seed": SEED,
    "window": WINDOW,
    "horizon": HORIZON,
    "features": FEATURES,
    "target": f"{TARGET}_delta",
    "test": test_metrics,
    "best_validation_rmse": best_val,
}
with open(RESULTS_DIR / "metrics.json", "w") as file:
    json.dump(metrics, file, indent=2)

pd.DataFrame(history).to_csv(RESULTS_DIR / "training_history.csv", index=False)
pd.DataFrame(
    {
        "actual": test_actuals,
        "prediction": test_preds,
        "residual": test_preds - test_actuals,
    }
).to_csv(RESULTS_DIR / "predictions.csv", index=False)

save_training_plots(history)
save_test_plots(test_actuals, test_preds)
print(f"Saved results to {RESULTS_DIR}")

Saved results to results/snn_v3
